# Instance Segmentation Fine-Tuning

Direct Graha/Lunar-FM true instance segmentation using TerraTorch `ObjectDetectionTask` with `framework="mask-rcnn"`.

## Config

In [ ]:
from argparse import Namespace
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "instance_seg_finetuning.ipynb").exists():
    NOTEBOOK_DIR = (Path.cwd() / "notebooks" / "full_model").resolve()
LFM_ROOT = NOTEBOOK_DIR.parents[1]

SIMLINK_DEST = None
DATA_ROOT = Path("/panfs/ccds02/nobackup/projects/lfm/model_inputs/300_300_inputs/full_model_inst_seg_v2")
BASE_OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "instance_seg_finetuning"
PRETRAIN_DIR = None
LIGHTNING_CHECKPOINT = None

CROP_SIZE = 256
STATS_BATCH_SIZE = 16
BATCH_SIZE = 2
NUM_WORKERS = 10
MAX_EPOCHS = 1

BACKBONE_LR = 5.0e-5
HEAD_LR = 2.0e-4
LAYER_DECAY = 0.75
WEIGHT_DECAY = 0.05
WARMUP_STEPS = 500
ANCHOR_SIZES = [[8], [16], [32], [64]]
ANCHOR_ASPECT_RATIOS = [0.5, 1.0, 2.0]
SCORE_THRESHOLD = 0.5
SEED = 42

RUN_FIT = False
LOSS_SMOKE_ONLY = False

## Environment

In [ ]:
import sys

for import_path in [NOTEBOOK_DIR, LFM_ROOT]:
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from lightning.pytorch import seed_everything

from lfm.full_model.utils import create_timestamped_output_dir
from lfm.full_model.utils.utils import ensure_data_symlink

import instance_seg_finetuning as workflow

workflow.configure_proj_environment()
ensure_data_symlink(SIMLINK_DEST, NOTEBOOK_DIR / "data")

## Build Config

In [ ]:
args = Namespace(
    simlink_dest=SIMLINK_DEST,
    data_root=str(DATA_ROOT) if DATA_ROOT is not None else None,
    base_output_dir=str(BASE_OUTPUT_DIR) if BASE_OUTPUT_DIR is not None else None,
    pretrain_dir=str(PRETRAIN_DIR) if PRETRAIN_DIR is not None else None,
    lightning_checkpoint=str(LIGHTNING_CHECKPOINT) if LIGHTNING_CHECKPOINT is not None else None,
    crop_size=CROP_SIZE,
    stats_batch_size=STATS_BATCH_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    max_epochs=MAX_EPOCHS,
    backbone_lr=BACKBONE_LR,
    head_lr=HEAD_LR,
    layer_decay=LAYER_DECAY,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    anchor_sizes=ANCHOR_SIZES,
    anchor_aspect_ratios=ANCHOR_ASPECT_RATIOS,
    score_threshold=SCORE_THRESHOLD,
    seed=SEED,
    no_fit=not RUN_FIT,
    loss_smoke_only=LOSS_SMOKE_ONLY,
)

config = workflow.build_config(args)
workflow.configure_python_paths(config)
workflow.print_config(config)
workflow.validate_required_paths(config)
deps = workflow.import_project_dependencies()

## Output Directory

In [ ]:
output_dir = create_timestamped_output_dir(config.base_output_dir)
(output_dir / "checkpoints" / "full_model").mkdir(parents=True, exist_ok=True)
workflow.save_config(config, output_dir)
seed_everything(config.seed)
print("Output directory:", output_dir)

## Training Stats

In [ ]:
datamodule_cls = deps["LunarObjectDetectionInstanceSegmentationDatamodule"]
means, stds = workflow.calculate_train_stats(config, datamodule_cls)

## Datamodule

In [ ]:
datamodule = workflow.create_datamodule(config, datamodule_cls, means, stds)
sample_batch = workflow.inspect_batch(datamodule)

## Graha Mask R-CNN Task

In [ ]:
task_cls = deps["LunarObjectDetectionTask"]
task = workflow.create_task(config, task_cls, sample_batch)
print(type(task.model))

## Loss Smoke Test

In [ ]:
workflow.run_loss_smoke(task, sample_batch)

## Train

In [ ]:
if RUN_FIT:
    trainer = workflow.create_trainer(config, output_dir)
    ckpt_path = str(config.lightning_checkpoint) if config.lightning_checkpoint is not None else None
    trainer.fit(task, datamodule=datamodule, ckpt_path=ckpt_path)
else:
    print("RUN_FIT is False; skipping trainer.fit().")